# Data Cleaning

Data cleaning is an important step before performing data analysis. Data can have missing values, duplicate rows, and columns stored as the wrong type. This notebook covers how to find and fix those problems with pandas.

In [20]:
import numpy as np
import pandas as pd

## 1. A messy dataset

Here's a small DataFrame with some of the problems you'll see in real data: missing values (`np.nan`), a duplicate row, and a numeric column stored as text.

In [21]:
students = pd.DataFrame({
    "name": ["Ada", "Sam", "Lin", "Theo", "Maya", "Sam"],
    "grade": [9, 10, 9, np.nan, 12, 10],
    "test_score": ["88", "72", "95", "60", None, "72"],
})
students

,name,grade,test_score
0,Ada,9.0,88
1,Sam,10.0,72
2,Lin,9.0,95
3,Theo,NaN,60
4,Maya,12.0,None
5,Sam,10.0,72


Notice: `Sam` appears twice with identical values (a duplicate row), `grade` has a missing value, and `test_score` looks numeric but is actually stored as text (notice the quotes).

## 2. Finding missing values

`.isna()` flags missing values cell by cell. Chain `.sum()` to count how many are missing per column.

In [22]:
print(students.isna())
print()
print(students.isna().sum())

    name  grade  test_score
0  False  False       False
1  False  False       False
2  False  False       False
3  False   True       False
4  False  False        True
5  False  False       False

name          0
grade         1
test_score    1
dtype: int64


**Exercise:** Build a small DataFrame (5-6 rows) about anything you like, with at least one missing value in two different columns. Use `.isna().sum()` to confirm where they are.

In [23]:
# Your code here
sports = pd.DataFrame( {
    "sport_name": ["Football", "Vollyball", "Basketball", "Handball", "Table tinnes", "Tinnes", "Chess", "Checker", "Table tinnes"],
    "sport_type": ["Physical", "Physical", None, "Physical", "Physical", "Physical", "Mental", "Mental", "Physical"],
    "number_of_players": ["22", "12", "10", "10", "2", np.nan, "2", "2", "2"]
})

print(sports.iloc[:])

print(sports.isna().sum())

     sport_name sport_type number_of_players
0      Football   Physical                22
1     Vollyball   Physical                12
2    Basketball       None                10
3      Handball   Physical                10
4  Table tinnes   Physical                 2
5        Tinnes   Physical               NaN
6         Chess     Mental                 2
7       Checker     Mental                 2
8  Table tinnes   Physical                 2
sport_name           0
sport_type           1
number_of_players    1
dtype: int64


## 3. Handling missing values

Two common options: drop rows with missing data (`.dropna()`), or fill them in with something sensible (`.fillna()`).

In [24]:
dropped = students.dropna()
print(dropped)

filled = students.copy()
filled["grade"] = filled["grade"].fillna(filled["grade"].median())
filled

  name  grade test_score
0  Ada    9.0         88
1  Sam   10.0         72
2  Lin    9.0         95
5  Sam   10.0         72


,name,grade,test_score
0,Ada,9.0,88
1,Sam,10.0,72
2,Lin,9.0,95
3,Theo,10.0,60
4,Maya,12.0,None
5,Sam,10.0,72


The right choice for data cleaning depends on the data. Dropping loses information, but filling in a guess can bias your results. Always think about *why* a value is missing before deciding.

**Exercise:** On the DataFrame you made above, try both approaches: drop the rows with missing values, and separately fill them in (with the mean, median, or a constant of your choice). Compare the two results.

In [33]:
# Your code here
dropped_sports = sports.dropna()
print(dropped_sports)

filled_sports = sports.copy()
filled_sports['number_of_players'] = filled_sports['number_of_players'].fillna(filled_sports['number_of_players'].median())
filled_sports['sport_type'] = filled_sports['sport_type'].fillna("Null")
filled_sports

     sport_name sport_type number_of_players
0      Football   Physical                22
1     Vollyball   Physical                12
3      Handball   Physical                10
4  Table tinnes   Physical                 2
6         Chess     Mental                 2
7       Checker     Mental                 2
8  Table tinnes   Physical                 2


,sport_name,sport_type,number_of_players
0,Football,Physical,22
1,Vollyball,Physical,12
2,Basketball,Null,10
3,Handball,Physical,10
4,Table tinnes,Physical,2
5,Tinnes,Physical,6.0
6,Chess,Mental,2
7,Checker,Mental,2
8,Table tinnes,Physical,2


## 4. Removing duplicate rows

`.duplicated()` flags rows that are exact repeats of an earlier row. `.drop_duplicates()` removes them.

In [34]:
print(students.duplicated())
students.drop_duplicates()

0    False
1    False
2    False
3    False
4    False
5     True
dtype: bool


,name,grade,test_score
0,Ada,9.0,88
1,Sam,10.0,72
2,Lin,9.0,95
3,Theo,NaN,60
4,Maya,12.0,None


**Exercise:** Add a duplicate row to your DataFrame on purpose (copy one of the existing rows). Confirm `.duplicated()` finds it, then remove it with `.drop_duplicates()`.

In [37]:
# Your code here
print(sports.duplicated())
sports.drop_duplicates()

0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8     True
dtype: bool


,sport_name,sport_type,number_of_players
0,Football,Physical,22
1,Vollyball,Physical,12
2,Basketball,None,10
3,Handball,Physical,10
4,Table tinnes,Physical,2
5,Tinnes,Physical,NaN
6,Chess,Mental,2
7,Checker,Mental,2


## 5. Fixing column types

`.dtypes` shows the type of every column. Numbers stored as text show up as `object`, which means you can't do math on them yet. Convert with `pd.to_numeric()`.

In [38]:
print(students.dtypes)

students["test_score"] = pd.to_numeric(students["test_score"])
print(students.dtypes)
students["test_score"].mean()

name           object
grade         float64
test_score     object
dtype: object
name           object
grade         float64
test_score    float64
dtype: object


77.4

Dates have the same problem: a column that looks like `"2024-01-15"` is just text until you convert it with `pd.to_datetime()`.

In [39]:
signup_dates = pd.DataFrame({"name": ["Ada", "Sam"], "signup": ["2024-01-15", "2024-03-02"]})
print(signup_dates.dtypes)

signup_dates["signup"] = pd.to_datetime(signup_dates["signup"])
print(signup_dates.dtypes)
signup_dates

name      object
signup    object
dtype: object
name              object
signup    datetime64[ns]
dtype: object


,name,signup
0,Ada,2024-01-15
1,Sam,2024-03-02


**Exercise:** Build a DataFrame with a numeric column stored as text (e.g. `"price": ["9.99", "14.50", "3.00"]`) and a date column stored as text. Check `.dtypes`, then convert both columns to their proper types.

In [56]:
# Your code here
product = pd.DataFrame({"name": ["Shipsy", "V-Cola", "SNICKERS"], "price": ["1.99", "2.50", "3.33"], "expiry_date": ["15-05-2027", "29-7-2027", "25/10/2026"]})

print(product.dtypes)

product['price'] = pd.to_numeric(product['price'])
product['expiry_date'] = pd.to_datetime(product['expiry_date'], format="mixed", dayfirst= True)

print(product.dtypes)
print(f"The average of prices: {product['price'].mean()}")
product


name           object
price          object
expiry_date    object
dtype: object
name                   object
price                 float64
expiry_date    datetime64[ns]
dtype: object
The average of prices: 2.606666666666667


,name,price,expiry_date
0,Shipsy,1.99,2027-05-15
1,V-Cola,2.50,2027-07-29
2,SNICKERS,3.33,2026-10-25


## 6. Mini Project

Now put it together. Pick **one**:

- Take the original `students` DataFrame from the top of this notebook and clean it fully: fix the missing `grade`, fix the `test_score` type, drop the duplicate, and end with a clean DataFrame you'd be happy to analyze.
- Grab a small real dataset from [Kaggle](https://www.kaggle.com/datasets) that has at least one missing value or wrong-typed column, load it with `pd.read_csv("filename.csv")`, and clean it: check `.isna().sum()`, check `.dtypes`, check for duplicates, and fix what you find.

In [74]:
# Your code here
students['grade'] = students['grade'].fillna(students['grade'].median())
students['test_score'] = pd.to_numeric(students['test_score'])
students['test_score'] = students['test_score'].fillna(students['test_score'].mean())
print(students.duplicated())
students = students.drop_duplicates()
students

0    False
1    False
2    False
3    False
4    False
dtype: bool


,name,grade,test_score
0,Ada,9.0,88.0
1,Sam,10.0,72.0
2,Lin,9.0,95.0
3,Theo,10.0,60.0
4,Maya,12.0,77.4
